# Basic information on the available datasets

Enumerate all malware files

In [ ]:
from pathlib import Path

folder = Path("../datasets/malware.parquet")  # change to Path("datasets") or any folder path
files = [p.name for p in folder.iterdir() if p.is_file()]
print(f"Files: len(files)")

['20240101_AGENTTESLA.parquet',
 '20240101_ARDAMAX.parquet',
 '20240101_ASYNCRAT.parquet',
 '20240101_AZORULT.parquet',
 '20240101_BLACKMOON.parquet',
 '20240101_COBALTSTRIKE.parquet',
 '20240101_CYBERGATE.parquet',
 '20240101_DARKCOMET.parquet',
 '20240101_DRIDEX.parquet',
 '20240101_FORMBOOK.parquet',
 '20240101_GULOADER.parquet',
 '20240101_HAWKEYE.parquet',
 '20240101_ICEDID.parquet',
 '20240101_LOKIBOT.parquet',
 '20240101_METASPLOIT.parquet',
 '20240101_MODILOADER.parquet',
 '20240101_NANOCORE.parquet',
 '20240101_NETWIRE.parquet',
 '20240101_NJRAT.parquet',
 '20240101_PONY.parquet',
 '20240101_RACCOON.parquet',
 '20240101_REMCOS.parquet',
 '20240101_REVENGERAT.parquet',
 '20240101_SALITY.parquet',
 '20240101_SMOKELOADER.parquet',
 '20240101_TRICKBOT.parquet',
 '20240101_UPATRE.parquet',
 '20240101_VIDAR.parquet',
 '20240101_WARZONERAT.parquet',
 '20240101_XMRIG.parquet',
 '20240101_XTREMERAT.parquet',
 '20240101_XWORM.parquet',
 '20240102_AGENTTESLA.parquet',
 '20240102_ARDAMAX.

Load all files to check their correctness

In [11]:
import pandas as pd
import pyarrow.parquet as pq
for file in files:
    print(f"File: {file}")
    schema = pq.read_schema(f"../datasets/malware.parquet/{file}")
    df = pd.read_parquet(f"../datasets/malware.parquet/{file}")
    print(schema)
    break
    print("Ok!")


File: 20240101_AGENTTESLA.parquet
bs: int64
ps: int64
br: int64
pr: int64
dp: int64
sp: int64
da: string
sa: string
ts: double
td: double
tls.cver: string
tls.sver: string
tls.sext: list<element: string>
  child 0, element: string
tls.csg: list<element: string>
  child 0, element: string
tls.ccs: list<element: string>
  child 0, element: string
tls.cext: list<element: string>
  child 0, element: string
tls.ssv: list<element: null>
  child 0, element: null
tls.csv: list<element: null>
  child 0, element: null
tls.scs: string
tls.alpn: list<element: string>
  child 0, element: string
tls.sni: string
tls.ja3: string
tls.ja4: string
tls.ja3s: string
tls.ja4s: string
meta.sample.id: string
meta.malware.family: string
meta.system.os: string
meta.system.service: string
meta.application.name: null
tls.rec: list<element: int64>
  child 0, element: int64
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 3855


In [9]:
from pathlib import Path
import pyarrow.parquet as pq
from collections import defaultdict

# Folder containing parquet part files
folder = Path("../datasets/malware.parquet")

# Collect observed type per column across files
types_by_col = defaultdict(set)
files_by_col_type = defaultdict(list)

for f in sorted(folder.glob("*.parquet")):
    schema = pq.read_schema(f)
    for field in schema:
        t = str(field.type)
        col = field.name
        types_by_col[col].add(t)
        files_by_col_type[(col, t)].append(f.name)

# Print columns that are specifically null vs string
print("Columns with both 'null' and 'string':")
found = False
for col, observed in sorted(types_by_col.items()):
    if "null" in observed and "string" in observed:
        found = True
        print(f"\n- {col}")
        print(f"  null   in {len(files_by_col_type[(col, 'null')])} file(s)")
        print(f"  string in {len(files_by_col_type[(col, 'string')])} file(s)")
        print("  example null files  :", files_by_col_type[(col, "null")][:5])
        print("  example string files:", files_by_col_type[(col, "string")][:5])

if not found:
    print("  none")

Columns with both 'null' and 'string':

- meta.malware.family
  null   in 6507 file(s)
  string in 10332 file(s)
  example null files  : ['20240101_COBALTSTRIKE.parquet', '20240101_DARKCOMET.parquet', '20240101_DRIDEX.parquet', '20240101_GULOADER.parquet', '20240101_HAWKEYE.parquet']
  example string files: ['20240101_AGENTTESLA.parquet', '20240101_ARDAMAX.parquet', '20240101_ASYNCRAT.parquet', '20240101_AZORULT.parquet', '20240101_BLACKMOON.parquet']

- meta.system.service
  null   in 47 file(s)
  string in 16792 file(s)
  example null files  : ['20240105_MATIEX.parquet', '20240308_QNODESERVICE.parquet', '20240331_DISCORDRAT.parquet', '20240418_MASSLOGGER.parquet', '20240418_MATIEX.parquet']
  example string files: ['20240101_AGENTTESLA.parquet', '20240101_ARDAMAX.parquet', '20240101_ASYNCRAT.parquet', '20240101_AZORULT.parquet', '20240101_BLACKMOON.parquet']

- tls.ja3s
  null   in 6 file(s)
  string in 16833 file(s)
  example null files  : ['20240428_ICEDID.parquet', '20240610_HAWKE

In [ ]:
folder = Path("../datasets/malware.parquet")
parts = [pd.read_parquet(p) for p in folder.glob("*AGENTTESLA.parquet")]
df_malware = pd.concat(parts, ignore_index=True)

In [13]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd


# ---------------------------
# Helpers
# ---------------------------

def _is_listlike_series(s: pd.Series) -> bool:
    """Heuristic: parquet list<...> becomes 'object' with python lists."""
    if s.dtype != "object":
        return False
    # sample a few non-null values
    sample = s.dropna().head(50).tolist()
    return any(isinstance(x, (list, tuple, np.ndarray)) for x in sample)

def _safe_value_counts(
    s: pd.Series, top_k: int = 10
) -> pd.DataFrame:
    vc = s.value_counts(dropna=False).head(top_k)
    return vc.rename_axis("value").reset_index(name="count")

def _missing_stats(s: pd.Series) -> Tuple[int, float]:
    n = len(s)
    missing = int(s.isna().sum())
    miss_rate = (missing / n) if n else float("nan")
    return missing, miss_rate

def _numeric_stats(s: pd.Series) -> Dict[str, Any]:
    x = pd.to_numeric(s, errors="coerce")
    x = x.dropna()
    if len(x) == 0:
        return {
            "min": np.nan, "max": np.nan, "mean": np.nan, "std": np.nan,
            "q50": np.nan, "q90": np.nan, "q95": np.nan, "q99": np.nan,
        }
    return {
        "min": float(x.min()),
        "max": float(x.max()),
        "mean": float(x.mean()),
        "std": float(x.std(ddof=1)) if len(x) > 1 else 0.0,
        "q50": float(x.quantile(0.50)),
        "q90": float(x.quantile(0.90)),
        "q95": float(x.quantile(0.95)),
        "q99": float(x.quantile(0.99)),
    }

def _string_stats(s: pd.Series) -> Dict[str, Any]:
    # treat empty string as missing-like (optional; keep separate)
    ss = s.astype("string")
    empty = int((ss == "").sum())
    non_empty = ss[(ss != "") & ss.notna()]
    nunique = int(non_empty.nunique(dropna=True))
    return {"n_unique_non_empty": nunique, "empty_count": empty}

def _flatten_list_series(s: pd.Series) -> pd.Series:
    """
    Flatten list column into a single Series of elements.
    Empty lists contribute nothing.
    """
    # Keep only list-like values; ignore scalars
    vals = []
    for x in s.dropna():
        if isinstance(x, (list, tuple, np.ndarray)):
            vals.extend(list(x))
        else:
            # sometimes list columns can have stray scalars; keep them too
            vals.append(x)
    return pd.Series(vals)

def _list_stats(s: pd.Series, top_k: int = 10) -> Dict[str, Any]:
    """
    Compute stats for list columns:
      - missing count (null list)
      - empty list count
      - avg list length, p95 length
      - unique element count
      - top-K elements
    """
    n = len(s)
    missing = int(s.isna().sum())
    # count empty lists
    empty_lists = 0
    lengths = []
    for x in s.dropna():
        if isinstance(x, (list, tuple, np.ndarray)):
            l = len(x)
            lengths.append(l)
            if l == 0:
                empty_lists += 1
        else:
            lengths.append(1)

    lengths_s = pd.Series(lengths, dtype="float") if lengths else pd.Series([], dtype="float")

    flat = _flatten_list_series(s)
    flat = flat.dropna()

    element_unique = int(flat.nunique(dropna=True)) if len(flat) else 0
    top_elements = _safe_value_counts(flat.astype("string"), top_k=top_k) if len(flat) else pd.DataFrame({"value": [], "count": []})

    out = {
        "missing_count": missing,
        "missing_rate": (missing / n) if n else float("nan"),
        "empty_list_count": empty_lists,
        "avg_list_len": float(lengths_s.mean()) if len(lengths_s) else np.nan,
        "p95_list_len": float(lengths_s.quantile(0.95)) if len(lengths_s) else np.nan,
        "max_list_len": float(lengths_s.max()) if len(lengths_s) else np.nan,
        "n_unique_elements": element_unique,
        "top_elements": top_elements,
    }
    return out


# ---------------------------
# Core column stats function
# ---------------------------

def column_statistics(
    df: pd.DataFrame,
    col: str,
    top_k: int = 10,
    treat_empty_string_as_missing: bool = False,
) -> Dict[str, Any]:
    """
    Compute per-column statistics:
      - dtype
      - missing counts/rates
      - unique counts
      - top-K values
      - numeric stats if numeric
      - list-element stats if list column
    """
    if col not in df.columns:
        return {"column": col, "present": False}

    s = df[col]
    n = len(s)
    missing, miss_rate = _missing_stats(s)

    out: Dict[str, Any] = {
        "column": col,
        "present": True,
        "dtype": str(s.dtype),
        "n_rows": int(n),
        "missing_count": missing,
        "missing_rate": float(miss_rate),
    }

    # List columns (e.g., tls.sext, tls.rec)
    if _is_listlike_series(s):
        lst = _list_stats(s, top_k=top_k)
        out.update({
            "kind": "list",
            "empty_list_count": lst["empty_list_count"],
            "avg_list_len": lst["avg_list_len"],
            "p95_list_len": lst["p95_list_len"],
            "max_list_len": lst["max_list_len"],
            "n_unique_elements": lst["n_unique_elements"],
        })
        # Save top elements separately
        out["top_elements"] = lst["top_elements"]
        return out

    # Numeric columns
    if pd.api.types.is_numeric_dtype(s):
        out["kind"] = "numeric"
        out["n_unique"] = int(s.nunique(dropna=True))
        out["top_values"] = _safe_value_counts(s, top_k=top_k)
        out.update(_numeric_stats(s))
        return out

    # Strings / categoricals / objects
    out["kind"] = "categorical"
    ss = s.astype("string")
    if treat_empty_string_as_missing:
        # treat "" as missing
        ss = ss.replace("", pd.NA)
        missing = int(ss.isna().sum())
        out["missing_count"] = missing
        out["missing_rate"] = float(missing / n) if n else float("nan")

    out.update(_string_stats(ss))
    out["n_unique"] = int(ss.dropna().nunique())
    out["top_values"] = _safe_value_counts(ss.fillna("<NA>"), top_k=top_k)
    return out


# ---------------------------
# Dataset-wide report
# ---------------------------

def dataset_statistics_report(
    df: pd.DataFrame,
    top_k: int = 10,
    treat_empty_string_as_missing: bool = False,
) -> Tuple[pd.DataFrame, Dict[str, pd.DataFrame]]:
    """
    Returns:
      1) summary_df: one row per column with scalar stats
      2) details: dict mapping column -> top-values df (or top-elements df for list cols)

    This is easy to export:
      summary_df.to_csv(...)
      details['tls.sext'].to_csv(...)
    """
    rows: List[Dict[str, Any]] = []
    details: Dict[str, pd.DataFrame] = {}

    for col in df.columns:
        st = column_statistics(
            df, col, top_k=top_k,
            treat_empty_string_as_missing=treat_empty_string_as_missing
        )

        # pull out heavy frames into details
        if st.get("kind") == "list" and isinstance(st.get("top_elements"), pd.DataFrame):
            details[col] = st.pop("top_elements")
        elif isinstance(st.get("top_values"), pd.DataFrame):
            details[col] = st.pop("top_values")

        rows.append(st)

    summary_df = pd.DataFrame(rows)

    # Make a consistent column order
    preferred = [
        "column", "present", "kind", "dtype", "n_rows",
        "missing_count", "missing_rate", "n_unique", "n_unique_non_empty", "empty_count",
        "avg_list_len", "p95_list_len", "max_list_len", "n_unique_elements",
        "min", "max", "mean", "std", "q50", "q90", "q95", "q99",
    ]
    cols = [c for c in preferred if c in summary_df.columns] + [c for c in summary_df.columns if c not in preferred]
    summary_df = summary_df[cols]

    return summary_df, details


# ---------------------------
# Optional: schema-specific convenience
# ---------------------------

MALWARE_SCHEMA_COLUMNS = [
    "bs", "ps", "br", "pr", "dp", "sp", "da", "sa", "ts", "td",
    "tls.cver", "tls.sver", "tls.sext", "tls.csg", "tls.ccs", "tls.cext",
    "tls.ssv", "tls.csv", "tls.scs", "tls.alpn", "tls.sni",
    "tls.ja3", "tls.ja4", "tls.ja3s", "tls.ja4s",
    "meta.sample.id", "meta.malware.family", "meta.system.os", "meta.system.service", "meta.application.name",
    "tls.rec",
]

def malware_dataset_report(
    df: pd.DataFrame,
    top_k: int = 10,
    treat_empty_string_as_missing: bool = False,
) -> Tuple[pd.DataFrame, Dict[str, pd.DataFrame]]:
    """
    Same as dataset_statistics_report but keeps a stable column order based on your schema
    and ignores unknown extra columns (if present).
    """
    cols_present = [c for c in MALWARE_SCHEMA_COLUMNS if c in df.columns]
    df2 = df[cols_present].copy()
    return dataset_statistics_report(df2, top_k=top_k, treat_empty_string_as_missing=treat_empty_string_as_missing)


In [ ]:
summary, details = malware_dataset_report(df_malware, top_k=15, treat_empty_string_as_missing=True)
print(summary)

01. Dataset sizes & time spans

The dataset provides three complementary sources of annotated TLS traffic:

| Dataset       | Source Environment | Size | Number of connections | Interval |
|---------------|--------------------|------|------------|-----------|
| `soho`        | Real network       | 104 files / 12.8MB | 108,036  | 2025-09-26 -- 2025-10-01 |
| `malware`     | Malware sandbox    | 16984 files / 1042.4 MB | ??? | 2024-01-01 -- 2025-09-30 |
| `winapps`     | Windows Sandbox    | 5844 files / 163.2 MB | ??? | 2024-07-16 -- 2025-04-02 |

